# Publications markdown generator for academicpages

Takes a set of bibtex of publications and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). 

The core python code is also in `pubsFromBibs.py`. 
Run either from the `markdown_generator` folder after replacing updating the publist dictionary with:
* bib file names
* specific venue keys based on your bib file preferences
* any specific pre-text for specific files
* Collection Name (future feature)

TODO: Make this work with other databases of citations, 
TODO: Merge this with the existing TSV parsing solution

In [10]:
from pybtex.database.input import bibtex
import pybtex.database.input.bibtex 
from time import strptime
import string
import html
import os
import re

In [11]:
#todo: incorporate different collection types rather than a catch all publications, requires other changes to template
publist = {
    #"proceeding": {
    #    "file" : "proceedings.bib",
    #    "venuekey": "booktitle",
    #    "venue-pretext": "In the proceedings of ",
    #    "collection" : {"name":"publications",
    #                    "permalink":"/publication/"}
        
    #},
    "journal":{
        "file": "./lijh0417.bib",
        "venuekey" : "journal",
        "venue-pretext" : "",
        "collection" : {"name":"publications",
                        "permalink":"/publication/"}
    } 
}

In [12]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    """Produce entities within text."""
    return "".join(html_escape_table.get(c,c) for c in text)

In [25]:
import os
import re
import html
from time import strptime
from pybtex.database.input import bibtex


def html_escape(text):
    return html.escape(str(text), quote=True)


def normalize_name(name):
    """统一作者名格式：去掉多余空格并转小写，便于匹配"""
    return " ".join(str(name).split()).strip().lower()


# 需要加粗显示的作者
highlight_authors = {normalize_name("JiaHao Li")}


for pubsource in publist:
    parser = bibtex.Parser()
    bibdata = parser.parse_file(publist[pubsource]["file"])

    # loop through the individual references in a given bibtex file
    for bib_id in bibdata.entries:
        pub_year = "1900"
        pub_month = "01"
        pub_day = "01"

        b = bibdata.entries[bib_id].fields

        try:
            # year
            pub_year = str(b.get("year", "1900")).strip()

            # month
            month_raw = str(b.get("month", "")).strip()
            if month_raw:
                if month_raw.isdigit():
                    pub_month = f"{int(month_raw):02d}"
                else:
                    try:
                        pub_month = f"{strptime(month_raw[:3], '%b').tm_mon:02d}"
                    except ValueError:
                        pub_month = "01"

            # day
            day_raw = str(b.get("day", "")).strip()
            if day_raw.isdigit():
                pub_day = f"{int(day_raw):02d}"

            pub_date = f"{pub_year}-{pub_month}-{pub_day}"

            # title
            raw_title = b.get("title", "Untitled")
            clean_title = (
                raw_title.replace("{", "")
                .replace("}", "")
                .replace("\\", "")
                .strip()
            )
            slug_title = clean_title.replace(" ", "-")
            url_slug = re.sub(r"\[.*?\]|[^a-zA-Z0-9_-]", "", slug_title)
            url_slug = re.sub(r"-+", "-", url_slug).strip("-")

            md_filename = f"{pub_date}-{url_slug}.md"
            html_filename = f"{pub_date}-{url_slug}"

            # authors
            authors = bibdata.entries[bib_id].persons.get("author", [])

            author_names_for_page = []
            citation_authors = []

            corresponding_raw = str(b.get("corresponding", "")).strip()
            corresponding_authors = [x.strip() for x in corresponding_raw.split(" and ") if x.strip()]

            for i, author in enumerate(authors):
                first = " ".join(author.first_names) if getattr(author, "first_names", None) else ""
                middle = " ".join(author.middle_names) if getattr(author, "middle_names", None) else ""
                prelast = " ".join(author.prelast_names) if getattr(author, "prelast_names", None) else ""
                last = " ".join(author.last_names) if getattr(author, "last_names", None) else ""
                lineage = " ".join(author.lineage_names) if getattr(author, "lineage_names", None) else ""

                full_name = " ".join(x for x in [first, middle, prelast, last, lineage] if x).strip()
                if not full_name:
                    continue

                normalized_full_name = normalize_name(full_name)

                # 页面作者列表里加粗
                if normalized_full_name in highlight_authors:
                    page_name = f"**{full_name}**"
                else:
                    page_name = full_name

                author_names_for_page.append(page_name)

                # citation 里也加粗
                if normalized_full_name in highlight_authors:
                    citation_authors.append(f"<strong>{full_name}</strong>")
                else:
                    citation_authors.append(full_name)

            authors_line = ", ".join(author_names_for_page)

            # citation authors
            citation = ""
            if citation_authors:
                citation = ", ".join(citation_authors) + ', '

            # citation title
            citation += "\"" + html_escape(clean_title) + ".\""

            # venue
            venuekey = publist[pubsource].get("venuekey", "journal")
            venue_pretext = publist[pubsource].get("venue-pretext", "")
            venue_raw = b.get(venuekey, "Unknown venue")
            venue = venue_pretext + venue_raw.replace("{", "").replace("}", "").replace("\\", "")

            volume = str(b.get("volume", "")).strip()
            number = str(b.get("number", "")).strip()
            pages = str(b.get("pages", "")).strip()

            if volume:
                venue += f" {volume}"
                if number:
                    venue += f"({number})"
            if pages:
                venue += f", {pages}"

            venue_bold = f"<strong>{html_escape(venue)}"

            citation += " " + venue_bold
            citation += ", " + pub_year + ".</strong>"

            # YAML
            md = "---\n"
            md += 'title: "' + html_escape(clean_title) + '"\n'
            md += "collection: " + publist[pubsource]["collection"]["name"]
            md += "\npermalink: " + publist[pubsource]["collection"]["permalink"] + html_filename

            note = False
            note_text = str(b.get("note", "")).strip()
            if len(note_text) > 5:
                md += "\nexcerpt: '" + html_escape(note_text) + "'"
                note = True

            md += "\ndate: " + str(pub_date)
            md += "\nvenue: '" + html_escape(venue) + "'"

            url = False
            url_text = str(b.get("url", "")).strip()
            if len(url_text) > 5:
                md += "\npaperurl: '" + url_text + "'"
                url = True

            md += "\ncitation: '" + citation + "'"
            md += "\n---\n\n"

            # page body
            if authors_line:
                md += "**Authors:** " + authors_line + "\n\n"

            if corresponding_authors:
                md += "**Corresponding author:** " + ", ".join(corresponding_authors) + "\n\n"

            if note:
                md += html_escape(note_text) + "\n\n"

            if url:
                md += '[Access paper here](' + url_text + '){:target="_blank"}\n'
            else:
                md += (
                    "Use [Google Scholar]"
                    "(https://scholar.google.com/scholar?q="
                    + html.escape(clean_title.replace(" ", "+"))
                    + '){:target="_blank"} for full citation\n'
                )

            md_filename = os.path.basename(md_filename)
            outdir = "../_publications"
            os.makedirs(outdir, exist_ok=True)

            with open(os.path.join(outdir, md_filename), "w", encoding="utf-8") as f:
                f.write(md)

            print(
                f'SUCCESSFULLY PARSED {bib_id}: "',
                raw_title[:60],
                "..." * (len(raw_title) > 60),
                '"'
            )

        except KeyError as e:
            print(
                f'WARNING Missing Expected Field {e} from entry {bib_id}: "',
                b.get("title", "NO TITLE")[:30],
                "..." * (len(b.get("title", "")) > 30),
                '"'
            )
            continue

        except IndexError as e:
            print(
                f'WARNING IndexError in entry {bib_id}: "',
                b.get("title", "NO TITLE")[:30],
                "..." * (len(b.get("title", "")) > 30),
                f'" -> {e}'
            )
            continue

        except Exception as e:
            print(
                f'WARNING Unexpected error in entry {bib_id}: {type(e).__name__} -> {e}'
            )
            continue

SUCCESSFULLY PARSED sun20233d: " 3D printing of thermosets with diverse rheological and funct ... "
SUCCESSFULLY PARSED qin2024biomimetic: " Biomimetic solar photocatalytic reactor for selective oxidat ... "
SUCCESSFULLY PARSED xiao2024bioinspired: " Bioinspired polysaccharide-based nanocomposite membranes wit ... "
SUCCESSFULLY PARSED chen2024hierarchical: " Hierarchical and reconfigurable interfibrous interface of bi ... "
SUCCESSFULLY PARSED li2024anisotropic: " Anisotropic fracture of two-dimensional Ta2NiSe5  "
SUCCESSFULLY PARSED li2025biomimetic: " Biomimetic turing machine: A multiscale theoretical framewor ... "
SUCCESSFULLY PARSED li2024strain: " Strain engineering of ion-coordinated nanochannels in nanoce ... "
SUCCESSFULLY PARSED qin2025bio: " A Bio-Inspired Magnetic Soft Robotic Fish for Efficient Sola ... "
SUCCESSFULLY PARSED song2025hygromechanical: " Hygromechanical deformation of wood cell walls regulated by  ... "
SUCCESSFULLY PARSED li2025harnessing: " Harnessing di